#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

#Business Transformation and Modelling

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_key AS customer_number,
    ci.firstname,
    ci.lastname,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.create_date AS createdate
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_key = ca.customer_number
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.customer_number
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

customer_key,customer_id,customer_number,firstname,lastname,country,marital_status,gender,birthdate,createdate
1,null,A01Ass,null,null,null,n/a,n/a,null,null
2,null,13451235,null,null,null,n/a,n/a,null,null
3,null,SF566,null,null,null,n/a,n/a,null,null
4,null,PO25,null,null,null,n/a,n/a,null,null
5,11000,AW00011000,Jon,Yang,Australia,MARRIED,MALE,1971-10-06,2025-10-06
6,11001,AW00011001,Eugene,Huang,Australia,SINGLE,MALE,1976-05-10,2025-10-06
7,11002,AW00011002,Ruben,Torres,Australia,MARRIED,MALE,1971-02-09,2025-10-06
8,11003,AW00011003,Christy,Zhu,Australia,SINGLE,FEMALE,1973-08-14,2025-10-06
9,11004,AW00011004,Elizabeth,Johnson,Australia,SINGLE,FEMALE,1979-08-05,2025-10-06
10,11005,AW00011005,Julio,Ruiz,Australia,SINGLE,MALE,1976-08-01,2025-10-06


#Write it to the Gold Table

In [0]:
(
  df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("gold.dim_customers")
)

#Sanity Checks for Gold Table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10

customer_key,customer_id,customer_number,firstname,lastname,country,marital_status,gender,birthdate,createdate
1,null,SF566,null,null,null,n/a,n/a,null,null
2,null,PO25,null,null,null,n/a,n/a,null,null
3,null,13451235,null,null,null,n/a,n/a,null,null
4,null,A01Ass,null,null,null,n/a,n/a,null,null
5,11000,AW00011000,Jon,Yang,Australia,MARRIED,MALE,1971-10-06,2025-10-06
6,11001,AW00011001,Eugene,Huang,Australia,SINGLE,MALE,1976-05-10,2025-10-06
7,11002,AW00011002,Ruben,Torres,Australia,MARRIED,MALE,1971-02-09,2025-10-06
8,11003,AW00011003,Christy,Zhu,Australia,SINGLE,FEMALE,1973-08-14,2025-10-06
9,11004,AW00011004,Elizabeth,Johnson,Australia,SINGLE,FEMALE,1979-08-05,2025-10-06
10,11005,AW00011005,Julio,Ruiz,Australia,SINGLE,MALE,1976-08-01,2025-10-06
